# Reproduce Project: Stability vs Drift in Credit Risk Modeling

This is the single canonical notebook for reproducing the current project workflow.

## What this notebook does
1. Clones the repo and installs dependencies
2. Mounts Google Drive
3. Loads and cleans the Lending Club dataset
4. Builds structured feature groups
5. Creates a temporal train/validation/test split
6. Prepares and exports a future LLM evaluation sample
7. Trains logistic regression, XGBoost, and MLP
8. Tunes thresholds on the validation year
9. Evaluates all three models on future test years
10. Saves final tables and plots to the checkpoint results folder

This notebook is intentionally thin. Reusable logic lives in `src/`.


## 0. Clone repo and install dependencies


In [ ]:

import os

REPO_NAME = "CS-4365"
REPO_URL = "https://github.com/ndave24/CS-4365.git"

%cd /content

if os.path.exists(f"/content/{REPO_NAME}"):
    !rm -rf /content/{REPO_NAME}

!git clone {REPO_URL} /content/{REPO_NAME}
%cd /content/CS-4365
!pip install -r requirements.txt


## 1. Mount Google Drive


In [ ]:

from google.colab import drive
drive.mount("/content/drive")


## 2. Set up imports


In [ ]:

import sys
from pathlib import Path

REPO_ROOT = Path("/content/CS-4365")
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

print("Repo root:", REPO_ROOT)
print("Exists:", REPO_ROOT.exists())


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

from src.load_data import load_and_clean_dataset, summarize_dataset
from src.preprocess import (
    PreprocessConfig,
    get_model_feature_groups,
    summarize_feature_groups,
)
from src.temporal_split import (
    TemporalTrainValTestSplitConfig,
    make_temporal_train_val_test_split,
    describe_split,
    describe_test_years,
)
from src.models.logistic import fit_logistic_pipeline
from src.models.xgboost_model import fit_xgboost_pipeline, XGBoostConfig
from src.models.mlp_model import fit_mlp_pipeline, MLPConfig
from src.evaluate import (
    evaluate_temporal_by_year,
    add_time_gap_column,
    save_temporal_metrics,
)
from src.thresholding import tune_threshold_on_validation
from src.llm_prep import (
    LLMPrepConfig,
    build_llm_eval_dataframe,
    export_llm_eval_csv,
    export_llm_eval_jsonl,
)


## 3. Configuration


In [ ]:

# Change to parquet if you prefer.
DATA_PATH = Path("/content/drive/MyDrive/datasets/lending_club.csv")

RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_END_YEAR = 2013
VAL_YEAR = 2014
TEST_YEARS = (2015, 2016, 2017, 2018)

PREPROCESS_CONFIG = PreprocessConfig(
    include_text=False,
    include_id=False,
    drop_zip_code=True,
    max_categorical_cardinality=50,
)

XGB_CONFIG = XGBoostConfig(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    eval_metric="logloss",
)

MLP_CONFIG = MLPConfig(
    svd_components=16,
    hidden_layer_sizes=(16, 8),
    activation="relu",
    alpha=0.0001,
    batch_size=1024,
    learning_rate_init=0.001,
    max_iter=20,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=3,
    random_state=42,
)

LLM_PREP_CONFIG = LLMPrepConfig(
    text_cols=("title", "desc"),
    structured_context_cols=(
        "loan_amnt",
        "term",
        "int_rate",
        "annual_inc",
        "dti",
        "fico_range_low",
        "fico_range_high",
        "purpose",
        "home_ownership",
        "emp_length",
    ),
    max_text_chars_per_field=500,
    include_structured_context=True,
    include_target=True,
)

print("DATA_PATH:", DATA_PATH)
print("Exists:", DATA_PATH.exists())
print("RESULTS_DIR:", RESULTS_DIR)


## 4. Load and clean dataset


In [ ]:

df = load_and_clean_dataset(DATA_PATH)
summarize_dataset(df)


## 5. Build structured feature groups


In [ ]:

feature_groups = get_model_feature_groups(df, config=PREPROCESS_CONFIG)
summarize_feature_groups(feature_groups)

print("Total feature columns:", len(feature_groups["feature_cols"]))
print("Numeric columns:", len(feature_groups["numeric_cols"]))
print("Categorical columns:", len(feature_groups["categorical_cols"]))


## 6. Create the temporal train/validation/test split

Final evaluation pipeline:
- train: years <= 2013
- validation: 2014
- test: 2015-2018


In [ ]:

split_config = TemporalTrainValTestSplitConfig(
    train_end_year=TRAIN_END_YEAR,
    val_year=VAL_YEAR,
    test_years=TEST_YEARS,
)

split_dict = make_temporal_train_val_test_split(
    df,
    config=split_config,
)

train_df = split_dict["train_df"]
val_df = split_dict["val_df"]
test_df = split_dict["test_df"]

print("Split summary")
display(describe_split(split_dict))

print("Test-year summary")
display(describe_test_years(test_df))


## 7. Prepare future LLM evaluation sample early

This does not run an LLM. It exports a prompt-ready sample for later checkpoints.


In [ ]:

llm_eval_df = build_llm_eval_dataframe(
    df=test_df,
    config=LLM_PREP_CONFIG,
    years=TEST_YEARS,
    sample_per_year=100,
    random_state=42,
)

display(llm_eval_df.head())
print("LLM eval sample shape:", llm_eval_df.shape)


In [ ]:

export_llm_eval_csv(
    llm_eval_df,
    RESULTS_DIR / "llm_eval_sample.csv",
)

export_llm_eval_jsonl(
    llm_eval_df,
    RESULTS_DIR / "llm_eval_sample.jsonl",
)


## 8. Notebook helpers


In [ ]:

def save_two_metric_plots(results_df, prefix, title_prefix):
    auc_f1_path = RESULTS_DIR / f"{prefix}_auc_f1_by_year.png"
    auc_f1_gap_path = RESULTS_DIR / f"{prefix}_auc_f1_by_time_gap.png"

    plt.figure(figsize=(8, 5))
    plt.plot(results_df["year"], results_df["auc"], marker="o", label="AUC")
    plt.plot(results_df["year"], results_df["f1"], marker="o", label="F1")
    plt.xlabel("Test Year")
    plt.ylabel("Score")
    plt.title(f"{title_prefix}: AUC and F1 by Test Year")
    plt.ylim(0.0, 1.0)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(auc_f1_path, dpi=200, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(results_df["time_gap"], results_df["auc"], marker="o", label="AUC")
    plt.plot(results_df["time_gap"], results_df["f1"], marker="o", label="F1")
    plt.xlabel("Time Gap")
    plt.ylabel("Score")
    plt.title(f"{title_prefix}: AUC and F1 by Time Gap")
    plt.ylim(0.0, 1.0)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(auc_f1_gap_path, dpi=200, bbox_inches="tight")
    plt.show()

    return auc_f1_path, auc_f1_gap_path


def run_tuned_model(model_name, fit_fn, train_df, val_df, test_df, feature_groups, fit_kwargs=None):
    fit_kwargs = fit_kwargs or {}

    model = fit_fn(
        train_df=train_df,
        feature_groups=feature_groups,
        **fit_kwargs,
    )

    best_row, threshold_df = tune_threshold_on_validation(
        model=model,
        val_df=val_df,
        feature_groups=feature_groups,
    )
    tuned_threshold = best_row["threshold"]

    results_df = evaluate_temporal_by_year(
        model=model,
        test_df=test_df,
        feature_groups=feature_groups,
        threshold=tuned_threshold,
    )
    results_df = add_time_gap_column(results_df, train_end_year=VAL_YEAR)
    results_df["model"] = model_name
    results_df["threshold"] = tuned_threshold

    save_temporal_metrics(results_df, RESULTS_DIR / f"temporal_metrics_{model_name}.csv")
    threshold_df.to_csv(RESULTS_DIR / f"{model_name}_validation_threshold_search.csv", index=False)
    save_two_metric_plots(results_df, prefix=model_name, title_prefix=model_name)

    return model, best_row, threshold_df, results_df


def plot_comparison_metric(results_df, x_col, metric_col, title, output_name):
    plt.figure(figsize=(8, 5))
    for model_name in results_df["model"].unique():
        model_df = results_df[results_df["model"] == model_name]
        plt.plot(model_df[x_col], model_df[metric_col], marker="o", label=model_name)

    plt.xlabel("Time Gap" if x_col == "time_gap" else "Test Year")
    plt.ylabel(metric_col.upper())
    plt.title(title)
    plt.ylim(0.0, 1.0)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    path = RESULTS_DIR / output_name
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    return path


## 9. Train, tune, and evaluate all three models


In [ ]:

logreg_model, logreg_best_row, logreg_threshold_df, logreg_results = run_tuned_model(
    model_name="logreg",
    fit_fn=fit_logistic_pipeline,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    feature_groups=feature_groups,
)

xgboost_model, xgboost_best_row, xgboost_threshold_df, xgboost_results = run_tuned_model(
    model_name="xgboost",
    fit_fn=fit_xgboost_pipeline,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    feature_groups=feature_groups,
    fit_kwargs={"config": XGB_CONFIG},
)

mlp_model, mlp_best_row, mlp_threshold_df, mlp_results = run_tuned_model(
    model_name="mlp",
    fit_fn=fit_mlp_pipeline,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    feature_groups=feature_groups,
    fit_kwargs={"config": MLP_CONFIG},
)


In [ ]:

best_thresholds_df = pd.DataFrame([
    {"model": "logreg", **logreg_best_row},
    {"model": "xgboost", **xgboost_best_row},
    {"model": "mlp", **mlp_best_row},
]).sort_values("model").reset_index(drop=True)

display(best_thresholds_df)
best_thresholds_df.to_csv(RESULTS_DIR / "best_thresholds.csv", index=False)


## 10. Final comparison tables


In [ ]:

comparison_df = pd.concat(
    [logreg_results, xgboost_results, mlp_results],
    ignore_index=True,
).sort_values(["model", "year"]).reset_index(drop=True)

display(comparison_df)

comparison_df.to_csv(
    RESULTS_DIR / "temporal_metrics_all_models.csv",
    index=False,
)


In [ ]:

summary_cols = ["model", "year", "threshold", "auc", "f1", "pred_positive_rate"]
summary_df = comparison_df[summary_cols].sort_values(["model", "year"]).reset_index(drop=True)

display(summary_df)
summary_df.to_csv(RESULTS_DIR / "summary_table.csv", index=False)


## 11. Final comparison plots


In [ ]:

plot_comparison_metric(
    comparison_df,
    x_col="year",
    metric_col="auc",
    title="AUC by Test Year",
    output_name="comparison_auc_by_year.png",
)

plot_comparison_metric(
    comparison_df,
    x_col="year",
    metric_col="f1",
    title="F1 by Test Year",
    output_name="comparison_f1_by_year.png",
)

plot_comparison_metric(
    comparison_df,
    x_col="time_gap",
    metric_col="auc",
    title="AUC by Time Gap",
    output_name="comparison_auc_by_time_gap.png",
)

plot_comparison_metric(
    comparison_df,
    x_col="time_gap",
    metric_col="f1",
    title="F1 by Time Gap",
    output_name="comparison_f1_by_time_gap.png",
)


## 12. Final output check


In [ ]:

print("Saved outputs to:", RESULTS_DIR)
for p in sorted(RESULTS_DIR.iterdir()):
    print("-", p.name)
